## Install and Import Libraries

In [1]:
!pip install pandas numpy scikit-learn -q

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## Download Data from GitHub

In [2]:
train_url = "https://raw.githubusercontent.com/AI-is-out-there/check-your-biomed-datascience-skills-review/main/dataset/task1/train.csv"
test_url = "https://raw.githubusercontent.com/AI-is-out-there/check-your-biomed-datascience-skills-review/main/dataset/task1/test_features.csv"

print("Downloading training data...")
train_df = pd.read_csv(train_url)
print(f"Training data loaded: {train_df.shape}")

print("Downloading test data...")
test_df = pd.read_csv(test_url)
print(f"Test data loaded: {test_df.shape}")

Training data loaded: (800, 19)
Test data loaded: (200, 18)


## Explore the Data

In [3]:
print("Training data columns:")
print(train_df.columns.tolist())

print("\nFirst few rows of training data:")
display(train_df.head())

print("\nTarget distribution:")
print(train_df['prediction'].value_counts())
print(f"\nTarget percentage:")
print(train_df['prediction'].value_counts(normalize=True) * 100)

Training data columns:
['patient_id', 'Возраст', 'Пол_мужской', 'ИМТ', 'Окружность_талии_см', 'САД_мм_рт_ст', 'ДАД_мм_рт_ст', 'Пульсовое_давление', 'Глюкоза_натощак_ммоль_л', 'HbA1c_%', 'ЛПНП_ммоль_л', 'ЛПВП_ммоль_л', 'Триглицериды_ммоль_л', 'СКФ_мл_мин', 'Курение', 'Физическая_активность_мин_нед', 'prediction', 'Статус_глюкозы', 'Доклинический_риск']

First few rows of training data:


,patient_id,Возраст,Пол_мужской,ИМТ,Окружность_талии_см,САД_мм_рт_ст,ДАД_мм_рт_ст,Пульсовое_давление,Глюкоза_натощак_ммоль_л,HbA1c_%,ЛПНП_ммоль_л,ЛПВП_ммоль_л,Триглицериды_ммоль_л,СКФ_мл_мин,Курение,Физическая_активность_мин_нед,prediction,Статус_глюкозы,Доклинический_риск
0,1,41,1,24.24,81.04,113.19,62.54,50.65,4.595,4.65,NaN,NaN,1.379,115.52,0,165.8,0,0,0
1,4,53,0,19.84,73.44,119.75,71.59,48.15,4.431,5.08,2.268,2.022,1.467,93.39,0,248.1,0,0,0
2,5,53,1,24.28,82.06,127.07,70.34,56.73,4.997,5.04,2.459,1.325,0.967,88.41,0,230.5,0,0,0
3,6,60,1,25.19,91.89,116.98,82.00,34.98,5.491,5.00,3.700,0.722,1.505,91.06,1,132.5,1,0,0
4,7,59,1,24.45,84.07,129.12,82.70,46.42,6.456,4.80,3.960,1.406,2.274,89.74,1,177.4,1,2,0



Target distribution:
prediction
0    655
1    145
Name: count, dtype: int64

Target percentage:
prediction
0    81.875
1    18.125
Name: proportion, dtype: float64


## Data Preprocessing

In [4]:
feature_cols = [
    'Возраст', 'ИМТ', 'Окружность_талии_см', 'САД_мм_рт_ст',
    'ДАД_мм_рт_ст', 'Глюкоза_натощак_ммоль_л', 'HbA1c_%',
    'ЛПНП_ммоль_л', 'Триглицериды_ммоль_л', 'СКФ_мл_мин',
    'Курение', 'Статус_глюкозы', 'Доклинический_риск'
]

for col in feature_cols:
    if col in train_df.columns:
        median_val = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_val)
        test_df[col] = test_df[col].fillna(median_val)

print("Missing values handled")
print(f"\nMissing values remaining in training data: {train_df[feature_cols].isnull().sum().sum()}")
print(f"Missing values remaining in test data: {test_df[feature_cols].isnull().sum().sum()}")

Missing values handled

Missing values remaining in training data: 0
Missing values remaining in test data: 0


## Rule-Based Classification Function

In [5]:
def predict_risk(row):
    """
    Risk-based prediction using medical thresholds
    Returns 1 for high risk, 0 for low risk
    """
    risk_score = 0

    if row['Возраст'] > 55:
        risk_score += 1

    if row['ИМТ'] > 30:
        risk_score += 1

    if row['Окружность_талии_см'] > 90:
        risk_score += 1

    if row['САД_мм_рт_ст'] > 140 or row['ДАД_мм_рт_ст'] > 90:
        risk_score += 1

    if row['Глюкоза_натощак_ммоль_л'] > 5.6:
        risk_score += 2

    if row['HbA1c_%'] > 5.7:
        risk_score += 2

    if row['ЛПНП_ммоль_л'] > 3.0:
        risk_score += 1

    if row['Триглицериды_ммоль_л'] > 1.7:
        risk_score += 1

    if row['СКФ_мл_мин'] < 60:
        risk_score += 1

    if row['Курение'] == 1:
        risk_score += 1

    if row['Статус_глюкозы'] > 0:
        risk_score += 2

    if row['Доклинический_риск'] > 0:
        risk_score += 1

    return 1 if risk_score >= 4 else 0

print("Risk prediction function defined")

Risk prediction function defined


## Validate on Training Data

In [6]:
train_predictions = train_df.apply(predict_risk, axis=1)

train_accuracy = (train_predictions == train_df['prediction']).mean()
print(f"Rule-based classifier training accuracy: {train_accuracy:.4f}")

print("\nPrediction vs Actual:")
comparison = pd.DataFrame({
    'actual': train_df['prediction'],
    'predicted': train_predictions
})
print(comparison.value_counts())

print("\nPer-class accuracy:")
for actual_class in [0, 1]:
    mask = train_df['prediction'] == actual_class
    class_accuracy = (train_predictions[mask] == train_df['prediction'][mask]).mean()
    print(f"Class {actual_class}: {class_accuracy:.4f}")

Rule-based classifier training accuracy: 0.7800

Prediction vs Actual:
actual  predicted
0       0            526
        1            129
1       1             98
        0             47
Name: count, dtype: int64

Per-class accuracy:
Class 0: 0.8031
Class 1: 0.6759


## Generate Test Predictions

In [7]:
test_predictions = test_df.apply(predict_risk, axis=1)

submission = pd.DataFrame({
    'patient_id': test_df['patient_id'],
    'prediction': test_predictions
})

print(f"Prediction distribution:")
print(submission['prediction'].value_counts())
print(f"\nPrediction percentage:")
print(submission['prediction'].value_counts(normalize=True) * 100)

print(f"\nFirst few predictions:")
display(submission.head(10))

Prediction distribution:
prediction
0    149
1     51
Name: count, dtype: int64

Prediction percentage:
prediction
0    74.5
1    25.5
Name: proportion, dtype: float64

First few predictions:


,patient_id,prediction
0,2,0
1,3,0
2,8,0
3,18,0
4,19,0
5,20,0
6,28,0
7,29,0
8,37,0
9,43,0


## Download Results

In [8]:
submission.to_csv('task1_predictions.csv', index=False)

print("Predictions saved to: task1_predictions.csv")
print(f"Total predictions: {len(submission)}")
print(f"Missing values: {submission.isnull().sum().sum()}")

from google.colab import files
files.download('task1_predictions.csv')

print("Task 1 completed successfully!")
print("Your predictions have been downloaded.")

Predictions saved to: task1_predictions.csv
Total predictions: 200
Missing values: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Task 1 completed successfully!
Your predictions have been downloaded.


## Alternative: Machine Learning Approach (Optional)

If you want to try a machine learning approach instead of rule-based classification, run this cell:

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report

X_train = train_df[feature_cols].copy()
y_train = train_df['prediction'].copy()
X_test = test_df[feature_cols].copy()

imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print("Training Random Forest classifier...")
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train_split, y_train_split)

val_predictions = model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_predictions)
print(f"Validation accuracy: {val_accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_val, val_predictions))

print("\nRetraining on full training data...")
model.fit(X_train_scaled, y_train)
ml_predictions = model.predict(X_test_scaled)

ml_submission = pd.DataFrame({
    'patient_id': test_df['patient_id'],
    'prediction': ml_predictions
})

print(f"\nML prediction distribution:")
print(ml_submission['prediction'].value_counts())

ml_submission.to_csv('task1_predictions_ml.csv', index=False)
files.download('task1_predictions_ml.csv')

print("\nML predictions saved and downloaded!")

Training Random Forest classifier...
Validation accuracy: 0.9437

Classification report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       131
           1       0.88      0.79      0.84        29

    accuracy                           0.94       160
   macro avg       0.92      0.89      0.90       160
weighted avg       0.94      0.94      0.94       160


Retraining on full training data...

ML prediction distribution:
prediction
0    174
1     26
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


ML predictions saved and downloaded!
